In [10]:
import os
import numpy as np
import matplotlib.pyplot as pl

In [11]:
os.chdir('C:\cygwin64\home\codes\dispatch2\data\eos\Tomida+Hori_2016')
import EOS
from scaling import cgs

In [3]:
eos=EOS.eos_t()

reading data from table tomida.dat


In [4]:
d=1e-12
T=100
P=eos.pressure(T,d)
U=eos.U(T,d)
P,U

(0.0035106052664206065, 8461278696.0848522)

##### Pressure units
The units of pressure is mass density times velocity squared = M L$^{-3}$ L$^2$ t$^{-2}$ = M L$^{-1}$ t$^{-2}$

1 Pa (Pascal) = 1 kg m$^{-1}$ s$^{-2}$ = $10^3$ g $10^{-2}$ cm$^{-1}$ s$^{-2}$ = $10$ cgs units (ba = barye)

1 bar = $10^5$ Pa = $10^6$ cgs units $\approx$ 1 atm

In [190]:
pl.figure(1)
pl.clf()
T=np.logspace(2.,4)
dd=(1e-12,1e-11,1e-10)
for d in dd:
    P=eos.pressure(T,d)
    s='d = {:5.0e}'.format(d)
    pl.loglog(P*1e-6,T,label=s)
    T1=5e3
    mu=1.25
    P1=d/cgs.m_u*cgs.k_b*T1/mu
    pl.loglog(P1*1e-6,T1,'o')
pl.legend()
pl.title('T(P) at constant density')
pl.ylabel('T [K]')
pl.xlabel('P [bar]');

To follow an adiabat we need to have $$\Delta U = \frac{P}{\rho}\Delta\ln\rho,$$ where 

$$\Delta U=U(T+\Delta T,\ln\rho+\Delta\ln\rho)-U(T,\ln\rho),$$ 

and since $U$ depends mainly on $T$, this may be approximated by

$$U \frac{\partial \ln U}{\partial \ln T} \Delta\ln T \approx  \frac{P}{\rho}\Delta\ln\rho,$$
or 
$$\frac{\Delta\ln T }{\Delta\ln\rho} \approx  \frac{P}{\rho U\frac{\partial \ln U}{\partial \ln T}},$$


Now that we have an adiabat, we can integrate the hydrostatic pressure balance equation, turning $\Delta\ln P$ into $\Delta \ln r$, where $r$ is radius:

$$\frac{dP}{dR}=-g\rho = - \frac{G M(R)\rho}{R^2} ,$$
so
$$\frac{d\ln P}{d\ln R}=-\frac{g\rho R}{P}=-\frac{G M(R)\rho}{R P},$$

$$\frac{d\ln R}{d\ln P}=-\frac{P}{g\rho R}=-\frac{R P}{G M(R)\rho},$$

Compute adiabats, with 200 points covering 5 decades in density, starting from 9 different (rho,T) points:

Note that this step may be skipped if it has been executed before; then proceed to the restore!

In [196]:
#adiabats={}
for T0 in [100]:
    for logd_min in [-10]:
        d=np.logspace(logd_min,logd_min+10,501)
        np.log10(d[1]/d[0])
        dlnd=np.log(d[1]/d[0])
        nd=len(d)
        T=np.zeros(nd)
        U=np.zeros(nd)
        P=np.zeros(nd)
        T[0]=T0
        U[0]=eos.U(T[0],d[0])
        P[0]=eos.pressure(T[0],d[0])

        for ir in range(1,nd):
            gamma=eos.gamma(T[ir-1],d[ir-1])
            T[ir]=T[ir-1]*np.exp(dlnd*(gamma-1.0))
            dlnT=1.0
            nit=0
            while(abs(dlnT) > 1e-4):
                U[ir]=eos.U(T[ir],d[ir])
                P[ir]=eos.pressure(T[ir],d[ir])
                d1=(d[ir]+d[ir-1])/2.0
                T1=(T[ir]+T[ir-1])/2.0
                U1=(U[ir]+U[ir-1])/2.0
                P1=(P[ir]+P[ir-1])/2.0
                dUdlnT=U1*eos.dlnUdlnT(T1,d1)
                dU=U[ir]-U[ir-1]
                Pd=(P[ir]+P[ir-1])/(d[ir]+d[ir-1])
                Eps=dU-Pd*dlnd
                dEpsdlnT=dUdlnT-P1/d1*dlnd
                dlnT=-Eps/dEpsdlnT
                T[ir]=T[ir]*np.exp(dlnT)
                nit+=1
            if ir%100==0:
                print(ir)
        ir=nd-1
        key='T{:2d}d-{:02d}'.format(T0,-logd_min)
        print('{} {:2d}  T:{:7.1f}  P:{:11.3e}  nit:{:2d}'.format(
            key,ir,T[ir],P[ir],nit))
        adiabats[key]={'T':T,'P':P,'U':U,'d':d}

50
100
150
200
250
300
350
400
450
500
T100d-10 500  T:141885.9  P:  9.644e+12  nit: 2


In [198]:
import save_restore as sr
sr.save('Jupiter2.save',adiabats)

Restore from this point when restarting

In [199]:
import save_restore as sr
adiabats=sr.restore('Jupiter2.save')
adiabats.keys()

dict_keys(['T100d-12', 'T100d-11', 'T100d-10'])

At small radii, where the density and pressure are nearly constant:
    
$$\frac{dP}{dR}= -\frac{G M(R)\rho}{R^2} = -\frac{4\pi G R \rho^2}{3},$$

$$\frac{d\ln P}{d R}= -\frac{4\pi G R \rho_0^2}{3 P_0},$$

$$\ln P= \ln P_0 - \frac{2\pi G R^2 \rho_0^2}{3 P_0},$$

So, if $P_0$ is the central pressure and $P_1$ is the next pressure we know on the adiabat, the corresponding radius, $R=R_1$, is given by

$$\frac{2\pi G R_1^2 \rho_0^2}{3 P_0} = \ln(P_0/P_1) $$

$$R_1= \left( \frac{3 P_0 \ln(P_0/P_1)}{2\pi G \rho_0^2} \right)^{1/2} $$

The dimension of ${1}/({G\rho_0})$ is time squared, and the dimension of $P_0/\rho_0$ is velocity squared, so dimensionally, the result is indeed a length.

In [200]:
i0=490
key='T100d-10'
v=adiabats[key]
P=v['P']
T=v['T']
d=v['d']
P0=P[i0]
T0=T[i0]
d0=d[i0]
P1=d[i0-1]
R1=np.sqrt(3*P0*np.log(P0/P1)/(2*np.pi*cgs.grav*d0**2))
print('Central temperature: {:.1f} K'.format(T0))
print('Central density: {:.2e} cgs'.format(d0))
print('The first point is at {:.1f} Earth radii'.format(R1/cgs.r_earth))
M1=4*np.pi*R1**3*d0/3
print('The first mass point is {:.1f} Earth masses'.format(M1/cgs.m_earth))
R2=(cgs.m_earth/d0)**(1./3.)
print('One Earth mass radius: {:.1f} Earth radii'.format(R2/cgs.r_earth))

Central temperature: 111736.8 K
Central density: 6.31e-01 cgs
The first point is at 80.1 Earth radii
The first mass point is 58753.9 Earth masses
One Earth mass radius: 3.3 Earth radii


Plot adiabats, marking central values

In [192]:
pl.figure(1); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('T [K]')
for k,v in adiabats.items():
    pl.loglog(v['d'],v['T'])
#pl.loglog(d0,T0,'o')
pl.savefig('Jupiter2_fig1.png')

pl.figure(2); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('P [bar]')
for k,v in adiabats.items():
    pl.loglog(v['d'],1e-6*v['P'])
#pl.loglog(d0,1e-6*P0,'o')
pl.savefig('Jupiter2_fig2.png')

pl.figure(3); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('U [erg/g]')
for k,v in adiabats.items():
    pl.loglog(v['d'],v['U'])
#pl.loglog(d0,v['U'][i0],'o')
pl.savefig('Jupiter2_fig3.png')

In [ ]:
for i in (4,5,6):
    pl.figure(i); pl.clf()
M=np.zeros(nd)

In [219]:
Me=[1099,598,355.6]
j=0
i0=444
for k,v in adiabats.items():
    if k[0:8]=='T100d-10':
        j+=1
        P=v['P']
        d=v['d']
        T=v['T']
        R=np.ones(nd)
        i=i0
        d0=d[i]
        R0=cgs.r_earth*2
        R[i]=R0
        M0=4*np.pi*R0**3/3*d0
        print('M0:',M0/cgs.m_earth,'R0:',R0/cgs.r_earth)
        M[i]=M0
        while(i > 0 and R[i] < cgs.r_earth*2500):
            i=i-1
            R[i]=R[i+1]
            M[i]=M[i+1]
            for nit in range(9):
                F=(R[i]*P[i]/(d[i]*M[i])+R[i+1]*P[i+1]/(d[i+1]*M[i+1]))/2.0
                dlnRdlnP=F/cgs.grav
                dlnP=np.log(P[i+1]/P[i])
                dlnR=dlnP*dlnRdlnP
                R[i]=R[i+1]*np.exp(dlnR)
                dM=4./3.*np.pi*(R[i]**3-R[i+1]**3)*(d[i]+d[i+1])/2.
                M[i]=M[i+1]+dM
                #print(i,nit,M[i]/cgs.m_earth,R[i]/cgs.r_earth)
        if i<0:
            i=0
        print('{:3d}  M:{:10.3e}  R:{:10.3e}  T:{:10.3e}  d:{:10.3e}'.\
              format(i, M[i]/cgs.m_earth, R[i]/cgs.r_earth,T[i],d[i]))

M0: 0.110073205122 R0: 2.0
  0  M: 9.614e+02  R: 7.538e+01  T: 1.000e+02  d: 1.000e-10


In [147]:
        print('    {:2d}{:8.3f}{:8.3f}{:8.3f}'.format(
            i-1,dlnRdlnP,R[i-1]/cgs.r_earth,M[i-1]/cgs.m_earth))
        #print('{:3d}{:14.3e}{:14.3e}{:14.3e}{:14.3e}{:14.3e}'.format(
        #      i-1,dlnRdlnP,d[i-1],T[i-1],R[i-1]/cgs.r_earth),
        #      M[i-1]/cgs.m_earth)
        title='Embryo mass: 350, 595, 1095'
        pl.figure(4)
        pl.xlabel('R/Re'); pl.ylabel('P [bar]')
        pl.title(title)
        pl.loglog(R[0:i]/cgs.r_earth,1e-6*P[0:i])
        pl.savefig('Jupiter_fig4.png')
        pl.figure(5)
        pl.xlabel('R/Re'); pl.ylabel('T [K]')
        pl.title(title)
        pl.loglog(R[0:i]/cgs.r_earth,T[0:i])
        pl.savefig('Jupiter_fig5.png')
        pl.figure(6)
        pl.xlabel('R/Re'); pl.ylabel('M [M_E]')
        pl.title(title)
        pl.loglog(R[0:i]/cgs.r_earth,M[0:i]/cgs.m_earth,'-+')
        pl.savefig('Jupiter_fig6.png')

M0 0.00114443630237 637100000.0 1.0
489 0 inf inf 1.38472708771e+21 6072.41765294 0.0648924495518 394.0540562
489 1 6.83457359776e+24 0.0 9.36970430027e+191 0.0 0.0648924495518 0.0
489 2 3.37349299584e+281 3.37349299584e+281 1.38472708771e+21 3036.20882647 0.0648924495518 197.0270281
489 3 6.83457359776e+24 0.0 2.53498150686e+106 2.25218392023e-168 0.0648924495518 1.46149731425e-169
489 4 3.37349299584e+281 3.37349299584e+281 1.38472708771e+21 3036.20882647 0.0648924495518 197.0270281
    4883036.209   0.000   0.000


__main__:29: RuntimeWarning: overflow encountered in double_scalars
C:\Users\Aake\Anaconda3\envs\yt\lib\site-packages\matplotlib\ticker.py:2241: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  "Data has no positive values, and therefore cannot be "
